# CogniForge Rack — cloud demo (Colab / base44)

This notebook exercises the CogniForge Rack without any cluster:

1. **stub-video** — the CPU-only placeholder generator from `modules/stub-video` produces a real `.mp4`.
2. **CogniForge GPU GEMM** — the *same* FP32 kernel that runs inside the emulated CogniForge GPU device (`provisioning/emulator/cogniforge-gemm.c`) is compiled with gcc and verified on this machine.

**base44:** open this file from the repo by its GitHub URL (base44 can pull notebooks straight from GitHub and run them). Just replace the `REPO` variable below with the correct repo if the default changes.

No GPU and no Docker required — a plain CPU is enough.

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/androidcircus/ManifestAI-cogniforge-vx.git"
RACK_DIR = os.path.join(os.getcwd(), "cogniforge-rack")

if not os.path.isdir(RACK_DIR):
    print(f"cloning {REPO}")
    subprocess.run(["git", "clone", "--depth", "1", REPO, RACK_DIR], check=True)
else:
    print(f"repo already present: {RACK_DIR}")

## Part A — stub-video generator (no GPU)

Install the tiny CPU-only deps, then run `generate.py` exactly the way the Rack Engine / Argo drives it: `python generate.py --image-url <seed> --duration <sec> --output <file>`.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", os.path.join(RACK_DIR, "modules", "stub-video", "requirements.txt")],
    check=True,
)

In [ ]:
OUT = os.path.join(os.getcwd(), "out.mp4")
gen = os.path.join(RACK_DIR, "modules", "stub-video", "generate.py")
r = subprocess.run(
    [sys.executable, gen, "--image-url", "https://example.com/cat.jpg",
     "--duration", "2", "--fps", "16", "--width", "448", "--height", "256",
     "--output", OUT],
    check=True,
)
print("bytes:", os.path.getsize(OUT))

In [ ]:
from IPython.display import Video, display

display(Video(OUT, embed=True, mimetype="video/mp4"))

## Part B — CogniForge GPU GEMM kernel (real C, compiled here)

`cogniforge-gemm.c` is the exact FP32 kernel emulated inside the CogniForge GPU device (a patched QEMU). It has **zero QEMU dependencies**, so it runs on any machine with gcc. We compile it and verify `C = A·B` numerically, including error codes for bad parameters.

In [ ]:
import shutil
assert shutil.which("gcc"), "gcc required for Part B"
print("gcc available:", shutil.which("gcc"))

In [ ]:
src = r'''
#include <stdio.h>
#include <string.h>
#include "cogniforge-gemm.h"

int main(void) {
    float A[4] = {1.0f, 2.0f, 3.0f, 4.0f};
    float B[4] = {5.0f, 6.0f, 7.0f, 8.0f};
    float C[4] = {0};
    CogniForgeGemmDesc d;
    memset(&d, 0, sizeof(d));
    d.m = d.n = d.k = 2;
    d.lda = d.ldb = d.ldc = 2;
    d.a = A; d.b = B; d.c = C;
    int rc = cogniforge_gemm(&d);
    printf("rc=%d C=[%.1f %.1f %.1f %.1f]\n", rc, C[0], C[1], C[2], C[3]);
    if (rc != 0 || C[0]!=19.0f || C[1]!=22.0f || C[2]!=43.0f || C[3]!=50.0f) return 1;

    d.m = 0;                              /* bad param -> -1 */
    int bad = cogniforge_gemm(&d);
    d.m = 2; d.n = 2; d.k = 2; d.lda = 2; d.ldb = 2; d.ldc = 2;
    d.m = COGNIFORGE_GEMM_MAX_DIM + 1;    /* too large -> -2 */
    int huge = cogniforge_gemm(&d);
    printf("bad=%d huge=%d\n", bad, huge);
    if (bad != -1 || huge != -2) return 1;
    puts("PASS: CogniForge GEMM verified (same kernel as the emulated GPU device)");
    return 0;
}
'''

emul = os.path.join(RACK_DIR, "provisioning", "emulator")
main_c = os.path.join(emul, "_colab_main.c")
with open(main_c, "w") as f:
    f.write(src)

bin_p = os.path.join(emul, "_colab_gemm")
subprocess.run(
    ["gcc", "-O2", "-Wall", main_c,
     os.path.join(emul, "cogniforge-gemm.c"), "-o", bin_p],
    check=True,
)
subprocess.run([bin_p], check=True)
os.remove(main_c); os.remove(bin_p)

## Next steps

- To see the *device-level* emulation (MMIO registers, VRAM, qtest): follow `provisioning/emulator/README.md`.
- For real Wan 2.1 14B inference: deploy the rack on NVIDIA GPUs via `docs/cloud-gpu.md` (tier-3 nodes, 40GB+ VRAM), then submit pipelines from the dashboard.
- To run this on **base44**: create a notebook session and import the file from the GitHub repo URL used above.

*Generated output:* `out.mp4` (downloaded from the Files sidebar).